# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² protein mass spectrometry dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library, leveraging Croissant metadata for reproducible scientific analysis.

### Dataset Source
The dataset schema is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` and required libraries are installed
!pip install mlcroissant pandas matplotlib

## 1. Data Loading
We load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Croissant schema URL
CROISSANT_URL = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata and object
dataset = mlc.Dataset(CROISSANT_URL)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\nPublished: {metadata.datePublished}")

## 2. Data Overview
Explore available record sets and their fields using their `@id` values (which uniquely identify them in the Croissant schema).

We will enumerate all record sets and fields. For the FAIR² dataset, the key record set is typically the main protein or peptide table.

In [ ]:
# List all available record sets and their field @ids

record_sets = dataset.metadata.recordSet
if hasattr(record_sets, '__iter__') and not isinstance(record_sets, str):
    print('RecordSets (@id, name, description):')
    for rs in record_sets:
        print(f"- @id: {rs['@id']}, name: {rs.get('name', 'N/A')}, description: {rs.get('description', 'N/A')}")
        # List fields for this record set
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print('  Fields:')
        for field in fields:
            print(f"    - @id: {field['@id']} | {field.get('name', 'N/A')} | type: {field.get('dataType', 'N/A')}")
else:
    print('No record sets listed in the schema.')

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. We use the `@id` of the protein-level table. For the FAIR² dataset, we identify the main record set by inspecting the output above.

**Note:** _Replace the variable `MAIN_RECORDSET_ID` with the actual `@id` printed above if necessary._

In [ ]:
# Set the main record set's @id as a variable (update as per printed schema overview)
# Example: MAIN_RECORDSET_ID = 'https://api.app.sen.science/frontiers/7154140/6a0af814-6b0a-438e-92b6-5efefa0967bd'
MAIN_RECORDSET_ID = '<FILL_THIS_IN_WITH_ACTUAL_RECORDSET_ID>'  # <-- Replace after inspecting previous output

if MAIN_RECORDSET_ID.startswith('<'):
    print('Please inspect the above output and set MAIN_RECORDSET_ID.')
else:
    # Extraction
    records = list(dataset.records(record_set=MAIN_RECORDSET_ID))
    df = pd.DataFrame(records)
    print(f"DataFrame loaded with {df.shape[0]} rows and {df.shape[1]} columns.")
    print('Fields:', df.columns.tolist())
    df.head()

## 4. Exploratory Data Analysis (EDA)
Let's perform example processing: filtering by a numeric field, normalizing that field, and grouping by a key attribute using `@id`-based access.

In [ ]:
# Example: choose field @id for molecular_weight (MW) and accession
# Update these based on schema, for example:
# MW_FIELD = 'MW' or '@id-of-molecular-weight-field'
# ACCESSION_FIELD = 'Accession' or '@id-of-accession-field'

MW_FIELD = '<FILL_THIS_IN_WITH_MW_FIELD_ID>'    # e.g. 'MW' or the field @id
ACCESSION_FIELD = '<FILL_THIS_IN_WITH_ACCESSION_ID>'  # e.g. 'Accession' or field @id

if MW_FIELD.startswith('<') or ACCESSION_FIELD.startswith('<'):
    print('Please set MW_FIELD and ACCESSION_FIELD to field names or @id values from the DataFrame.')
elif MW_FIELD not in df.columns:
    print(f"Column '{MW_FIELD}' not found in DataFrame. Available columns:", df.columns.tolist())
else:
    # Filtering out proteins with MW > threshold
    threshold = 10000
    filtered_df = df[df[MW_FIELD].astype(float) > threshold].copy()
    print(f"Filtered {len(filtered_df)} records with {MW_FIELD} > {threshold}.")

    # Normalize MW
    filtered_df[f"{MW_FIELD}_normalized"] = (
        (filtered_df[MW_FIELD].astype(float) - filtered_df[MW_FIELD].astype(float).mean()) /
        filtered_df[MW_FIELD].astype(float).std()
    )
    print(filtered_df[[ACCESSION_FIELD, MW_FIELD, f"{MW_FIELD}_normalized"]].head())

    # Group by another field if available (e.g. 'Description' or experimental group)
    GROUP_FIELD = '<FILL_THIS_IN_WITH_GROUP_FIELD_ID>'  # e.g. another @id or name
    if not GROUP_FIELD.startswith('<') and GROUP_FIELD in filtered_df.columns:
        grouped_df = filtered_df.groupby(GROUP_FIELD)[MW_FIELD].mean().reset_index()
        print(f"Grouped mean {MW_FIELD} by {GROUP_FIELD} (top rows):")
        print(grouped_df.head())

## 5. Visualization
Visualize the protein molecular weight distribution (or another numeric column) as a histogram.

In [ ]:
if not MW_FIELD.startswith('<') and MW_FIELD in df.columns:
    plt.figure(figsize=(8, 5))
    df[MW_FIELD].astype(float).hist(bins=30)
    plt.xlabel('Molecular Weight (Da)')
    plt.ylabel('Count')
    plt.title('Distribution of Protein Molecular Weight')
    plt.show()
else:
    print('Please set MW_FIELD to a valid numeric column to plot.')

## 6. Conclusion
In this notebook, we demonstrated loading, overview, extraction, and basic EDA of a FAIR² protein mass spectrometry dataset using `mlcroissant`. By referencing all key elements using their Croissant `@id`, this workflow is reproducible and data model-compliant. Adjust the code above to focus on additional attributes and other available record sets for deeper analysis.